In [1]:
import json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO  = Path("/home/g0amer/Desktop/thesis/01_Task_Recognition_Multimodal_Fusion")
OUT   = REPO / "output/research_outputs/fusion_training/revision_8020"
PAPER = REPO / "paper"

# Every value plotted here is read from the saved result tables of the training
# notebook. Nothing is recomputed or retyped, so a figure cannot drift from the
# table it illustrates.
head = pd.read_csv(OUT / "headline_8020_clean.csv")
ben  = pd.read_csv(OUT / "fusion_benefit_8020_clean.csv")
lat  = pd.read_csv(OUT / "latency_8020_clean.csv")

# Palette: slots 1-3 of the validated categorical set, checked on a white paper
# surface. Lightness band, chroma floor, CVD separation and the normal-vision floor
# all pass; the aqua slot sits below 3:1 contrast, which the relief rule covers
# because every plotted value also appears in a numbered table.
S1, S2, S3 = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#d8d7d2"
SHORT = {"Logistic Regression": "LogReg", "Random Forest": "Random Forest",
         "MLP": "MLP", "XGBoost": "XGBoost", "Two-Tower Fusion": "Two-Tower"}
print("headline", len(head), "| fusion", len(ben), "| latency", len(lat))

def tidy(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID); ax.spines[s].set_linewidth(0.6)
    ax.tick_params(labelsize=6.3, colors=INK2, length=2, width=0.6)
    ax.set_axisbelow(True)


headline 5 | fusion 4 | latency 6


In [2]:
# ---- Detector comparison, placed in the Detector Comparison subsection --------
# One panel per figure. A two-panel figure spanning two subsections cannot sit in
# the exact position of either, which is why each chart now travels alone. The
# in-axes title is dropped because the LaTeX caption below the float names it.
fig, ax1 = plt.subplots(figsize=(4.8, 2.15), dpi=400)

h = head.copy(); h["short"] = h["model"].map(SHORT)
h = h.sort_values("macro_f1")                  # ascending, so the best sits on top
y = np.arange(len(h)); bh = 0.26
for k, (col, colour, lab) in enumerate([("accuracy", S1, "Accuracy"),
                                        ("balanced_acc", S2, "Balanced acc."),
                                        ("macro_f1", S3, "Macro-F1")]):
    ax1.barh(y + (1 - k) * bh, h[col], height=bh * 0.9, color=colour, label=lab,
             edgecolor="white", linewidth=0.35, zorder=3)
ax1.set_yticks(y); ax1.set_yticklabels(h["short"], fontsize=6.5, color=INK)
ax1.set_xlim(0.5, 1.01); ax1.set_xticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
ax1.set_ylim(-0.6, len(h) - 0.4)
ax1.xaxis.grid(True, color=GRID, linewidth=0.5, zorder=0)
ax1.set_xlabel("score", fontsize=6.4, color=INK2)
ax1.legend(fontsize=6.1, frameon=False, ncol=3, loc="lower left",
           bbox_to_anchor=(0.0, 1.01, 1.0, 0.10), mode="expand",
           handlelength=1.0, handleheight=0.7, borderpad=0.0, columnspacing=1.0)
tidy(ax1)
fig.subplots_adjust(left=0.155, right=0.985, top=0.80, bottom=0.175)
fig.savefig(PAPER / "results_detectors.png", dpi=400)
plt.close(fig)
print("wrote paper/results_detectors.png")


wrote paper/results_detectors.png


In [3]:
# ---- Fusion gain, placed in the Does Fusion Help subsection -------------------
fig, ax2 = plt.subplots(figsize=(4.8, 1.85), dpi=400)

b = ben.sort_values("delta")                   # largest gain on top
yb = np.arange(len(b))
ax2.hlines(yb, b["single_macro_f1"], b["fused_macro_f1"], color=GRID,
           linewidth=1.4, zorder=2)
ax2.scatter(b["single_macro_f1"], yb, s=26, color=S2, zorder=3,
            edgecolor="white", linewidth=0.6, label="Best single modality")
ax2.scatter(b["fused_macro_f1"], yb, s=26, color=S1, zorder=3,
            edgecolor="white", linewidth=0.6, label="Fused input")
for yy, s, f, d in zip(yb, b["single_macro_f1"], b["fused_macro_f1"], b["delta"]):
    ax2.annotate(f"+{d:.3f}", (f, yy), textcoords="offset points", xytext=(5, 0),
                 va="center", fontsize=6.0, color=INK2)
ax2.set_yticks(yb); ax2.set_yticklabels([SHORT[m] for m in b["model"]],
                                        fontsize=6.5, color=INK)
ax2.set_xlim(0.40, 1.05); ax2.set_xticks([0.4, 0.6, 0.8, 1.0])
ax2.set_ylim(-0.7, len(b) - 0.3)
ax2.xaxis.grid(True, color=GRID, linewidth=0.5, zorder=0)
ax2.set_xlabel("macro-F1", fontsize=6.4, color=INK2)
ax2.legend(fontsize=6.1, frameon=False, ncol=2, loc="lower left",
           bbox_to_anchor=(0.0, 1.01, 1.0, 0.10), mode="expand",
           handlelength=1.0, handleheight=0.7, borderpad=0.0, columnspacing=1.0)
tidy(ax2)
fig.subplots_adjust(left=0.155, right=0.985, top=0.755, bottom=0.205)
fig.savefig(PAPER / "results_fusion.png", dpi=400)
plt.close(fig)
print("wrote paper/results_fusion.png")


wrote paper/results_fusion.png


In [4]:
# ---- Inference cost, placed in the Real-Time Feasibility subsection -----------
# Detectors are compared against each other only. The hop reference is gone, so the
# axis spans the measured range rather than the distance to a budget that no model
# comes close to, and the log scale carries the two orders of magnitude that separate
# the cheapest detector from the most expensive.
fig, ax3 = plt.subplots(figsize=(4.8, 1.95), dpi=400)

d = lat[~lat["model"].str.startswith("Window hop")].sort_values("latency_us")
yd = np.arange(len(d))
ax3.hlines(yd, 1.0, d["latency_us"], color=GRID, linewidth=1.1, zorder=2)
ax3.scatter(d["latency_us"], yd, s=26, color=S1, zorder=3,
            edgecolor="white", linewidth=0.6)
for yy, v in zip(yd, d["latency_us"]):
    ax3.annotate(f"{v:.1f}", (v, yy), textcoords="offset points", xytext=(5, 0),
                 va="center", fontsize=6.0, color=INK2)
ax3.set_xscale("log"); ax3.set_xlim(1.0, 1000.0)
ax3.set_xticks([1, 10, 100, 1000])
ax3.set_yticks(yd); ax3.set_yticklabels([SHORT[m] for m in d["model"]],
                                        fontsize=6.5, color=INK)
ax3.set_ylim(-0.65, len(d) - 0.25)
ax3.set_xlabel("microseconds per window, log scale", fontsize=6.4, color=INK2)
ax3.xaxis.grid(True, color=GRID, linewidth=0.5, zorder=0)
tidy(ax3)
fig.subplots_adjust(left=0.175, right=0.985, top=0.945, bottom=0.215)
fig.savefig(PAPER / "results_latency.png", dpi=400)
plt.close(fig)
print("wrote paper/results_latency.png")


wrote paper/results_latency.png


In [5]:
from PIL import Image
for name in ("results_detectors.png", "results_fusion.png", "results_latency.png"):
    im = Image.open(PAPER / name)
    print(f"{name}: {im.size[0]/400:.2f} x {im.size[1]/400:.2f} in at 400 dpi, "
          f"included at columnwidth 4.80 in")


results_detectors.png: 4.80 x 2.15 in at 400 dpi, included at columnwidth 4.80 in
results_fusion.png: 4.80 x 1.85 in at 400 dpi, included at columnwidth 4.80 in
results_latency.png: 4.80 x 1.95 in at 400 dpi, included at columnwidth 4.80 in
